In [ ]:
import pandas as pd

sp = pd.read_csv("../data/external/sponsors.csv")
print(sp.shape)
print(sp.columns.tolist())
print(sp.head(5).to_string())

In [ ]:
print(sp["Route"].value_counts())
print()
print(sp["Organisation Name"].nunique())

In [ ]:
skilled = sp[sp["Route"] == "Skilled Worker"].copy()
print(len(skilled))

meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
print(meta["company_name"].nunique())

direct = set(meta["company_name"]) & set(skilled["Organisation Name"])
print("直接匹配:", len(direct))
print(list(direct)[:20])

In [ ]:
import re

SUFFIXES = r"\b(?:limited|ltd|llp|plc|inc|incorporated|corp|corporation|" \
           r"company|co|group|holdings|uk|gb|international|services|" \
           r"solutions|technologies|technology)\b"

def norm(name):
    if not isinstance(name, str):
        return ""
    s = name.lower()
    s = re.sub(r"\bt/a\b.*", "", s)        # 去掉 "trading as" 及其后内容
    s = s.replace("&", "and")
    s = re.sub(r"[^\w\s]", " ", s)          # 标点转空格
    s = re.sub(SUFFIXES, " ", s)            # 去掉公司后缀
    s = re.sub(r"\s+", " ", s).strip()      # 压缩空白
    return s

for t in ["Robert Half Limited", "BRANOS OXFORD LTD T/A LILO",
          "Deliveroo", "Anson Mccade", "Anson McCade",
          "IT Online Learning", "Turner & Townsend"]:
    print(f"{t:35s} → {norm(t)}")

In [ ]:
skilled["norm"] = skilled["Organisation Name"].apply(norm)
meta["norm"] = meta["company_name"].apply(norm)

sponsor_set = set(skilled["norm"]) - {""}
print("名单归一化后不重复:", len(sponsor_set))

companies = meta[["company_name", "norm"]].drop_duplicates("company_name")
companies["licensed"] = companies["norm"].isin(sponsor_set)

print("我的公司数:", len(companies))
print("匹配上:", companies["licensed"].sum())
print("匹配率:", round(companies["licensed"].mean() * 100, 1), "%")

In [ ]:
matched = companies[companies["licensed"]]["company_name"].tolist()
unmatched = companies[~companies["licensed"]]["company_name"].tolist()

print("=== 匹配上的前 30 ===")
for c in matched[:30]:
    print(" ", c)
print()
print("=== 没匹配上的前 30 ===")
for c in unmatched[:30]:
    print(" ", c)

In [ ]:
AGENCY_PAT = r"\b(?:recruit\w*|recruiting|staffing|resourc\w*|talent|" \
             r"search|selection|consultan\w*|associates|partners|" \
             r"personnel|placements?|headhunt\w*)\b"

companies["is_agency"] = companies["company_name"].str.lower().str.contains(AGENCY_PAT, regex=True)

print(pd.crosstab(companies["is_agency"], companies["licensed"]))
print()
print("中介占比:", round(companies["is_agency"].mean() * 100, 1), "%")

In [ ]:
meta2 = meta.merge(companies[["company_name", "licensed", "is_agency"]],
                   on="company_name", how="left")
print("按岗位数：")
print(pd.crosstab(meta2["is_agency"], meta2["licensed"], normalize=True).round(3) * 100)

In [ ]:
companies["norm_len"] = companies["norm"].str.split().str.len()
short = companies[companies["licensed"] & (companies["norm_len"] <= 1)]
print("单词名匹配上的:", len(short))
print(short["company_name"].tolist())

In [ ]:
for c in short["company_name"].head(8):
    n = norm(c)
    hits = skilled[skilled["norm"] == n]["Organisation Name"].unique()[:4]
    print(f"{c}  →  {n}")
    for h in hits:
        print("      ", h)
    print()


In [ ]:
COMMON_WORDS = {
    "wise", "beyond", "trace", "salt", "fin", "dex", "betty", "prophet",
    "ki", "gamma", "sky", "snap", "google", "amazon", "citi", "swift",
    "apex", "howden", "markel", "qa", "rsm", "ing", "thg", "bp",
}

def confidence(row):
    if not row["licensed"]:
        return "unmatched"
    n = row["norm"]
    if len(n) <= 3:
        return "low"
    if " " not in n and n in COMMON_WORDS:
        return "low"
    if " " not in n and len(n) <= 6:
        return "medium"
    return "high"

companies["confidence"] = companies.apply(confidence, axis=1)
print(companies["confidence"].value_counts())

In [ ]:
meta3 = meta.merge(
    companies[["company_name", "licensed", "is_agency", "confidence"]],
    on="company_name", how="left"
)

print("按岗位数：")
print(meta3["confidence"].value_counts())
print()
print((meta3["confidence"].value_counts(normalize=True) * 100).round(1))
print()
print("非中介 + high:", ((meta3["confidence"] == "high") & (~meta3["is_agency"])).sum())

In [ ]:
out = meta3[["id", "title", "company_name", "region", "licensed",
             "confidence", "is_agency"]]
out.to_csv("../data/processed/sponsor_match_20260812.csv", index=False)
print(out.shape)